# 03 — Modeling and Evaluation

This notebook first audits the Phase 2 encoding table, applies the documented high-cardinality correction, and verifies a reasonable feature count. It then trains the required class-balanced Logistic Regression and Random Forest pipelines on a fixed stratified split. SHAP and later-phase explainability are not performed here.

## Setup and encoding audit — before training

The original Phase 2 decision table expanded two fields beyond the 20-level limit. A raw high-cardinality identifier is removed because its codes have no numerical meaning; the other high-cardinality category is frequency encoded using training-set prevalence without reading the target. The original audit is persisted so repeat runs do not lose this decision history.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import (
    audit_existing_encoding_decisions, build_model_pipelines,
    prepare_training_data, verify_corrected_preprocessing
)
from src.evaluate import (
    evaluate_trained_models, render_modeling_summary, save_best_model,
    save_comparison, save_confusion_matrices
)

results_dir = PROJECT_ROOT / 'results'
data = prepare_training_data(PROJECT_ROOT / 'data', results_dir / 'dataset_summary.txt')
audit = audit_existing_encoding_decisions(
    data['cleaned'],
    results_dir / 'tables' / 'encoding_decisions.csv',
    results_dir / 'tables' / 'encoding_correction_audit.csv'
)
print('High-cardinality audit completed before training:')
print(audit.to_string(index=False))

High-cardinality audit completed before training:
    column  unique_values previous_encoding  previous_output_columns         correction                                                                                       reason
product_id            899           one-hot                      899               drop raw high-cardinality identifier has no stable ordinal meaning and may encourage memorisation
  location            225           one-hot                      225 frequency encoding                retain prevalence information in one numeric feature without using the target


## Confirm the corrected shared preprocessing

The notebook reuses `identify_feature_roles` and `build_model_preprocessors` indirectly through the Phase 2 module. It does not create a separate encoding pipeline. A hard guard stops execution if a one-hot field still has more than 20 levels or if either design matrix reaches 300 features.

In [2]:
pd.DataFrame(data['roles']['decisions']).to_csv(
    results_dir / 'tables' / 'encoding_decisions.csv', index=False
)
feature_counts = verify_corrected_preprocessing(data['X_train'], data['roles'])
corrected_decisions = pd.DataFrame(data['roles']['decisions'])
print('Corrected decisions for audited fields:')
print(corrected_decisions[corrected_decisions['column'].isin(audit['column'])].to_string(index=False))
print('\nVerified feature counts:', feature_counts)

Corrected decisions for audited fields:
    column                         role  encoding                                                                                                 reason
product_id          excluded identifier      none              identifier-like field with 899 unique values (3.60% of rows) could encourage memorisation
  location high-cardinality categorical frequency 225 observed labels exceed the 20-level one-hot limit; one training-frequency feature avoids expansion

Verified feature counts: {'tree': 52, 'linear': 52}


## Fixed stratified train/test split

A 20% held-out test set is created with `random_state=42` and `stratify=y`. Stratification preserves the approximately 77/23 target distribution in both partitions.

In [3]:
print('Training shape:', data['X_train'].shape)
print('Test shape:', data['X_test'].shape)
split_balance = pd.DataFrame({
    'train_count': data['y_train'].value_counts().sort_index(),
    'train_percent': data['y_train'].value_counts(normalize=True).sort_index() * 100,
    'test_count': data['y_test'].value_counts().sort_index(),
    'test_percent': data['y_test'].value_counts(normalize=True).sort_index() * 100
})
print(split_balance.to_string(float_format=lambda value: f'{value:.2f}'))

Training shape: (20000, 28)
Test shape: (5000, 28)
           train_count  train_percent  test_count  test_percent
purchased                                                      
0                15507          77.53        3877         77.54
1                 4493          22.46        1123         22.46


## Logistic Regression baseline

The baseline receives one-hot/ordinal/frequency encoding and standardized numeric fields. `class_weight='balanced'` gives the minority purchase class more influence during fitting.

In [4]:
pipelines = build_model_pipelines(data['roles'])
trained_models = {}
print('Training Logistic Regression...')
trained_models['Logistic Regression'] = pipelines['Logistic Regression'].fit(
    data['X_train'], data['y_train']
)
print('Logistic Regression training complete.')

Training Logistic Regression...


Logistic Regression training complete.


## Random Forest main model

The forest receives the same corrected categorical representation but leaves numeric measurements unscaled. It also uses balanced class weights. A single worker and minimum leaf size of two keep training reproducible on ordinary student hardware.

In [5]:
print('Training Random Forest...')
trained_models['Random Forest'] = pipelines['Random Forest'].fit(
    data['X_train'], data['y_train']
)
print('Random Forest training complete.')
print('XGBoost skipped: it is outside the project-wide allowed dependency list; no third model was forced.')

Training Random Forest...


Random Forest training complete.
XGBoost skipped: it is outside the project-wide allowed dependency list; no third model was forced.


## Held-out evaluation

Accuracy, precision, recall, F1, and ROC-AUC come only from predictions on the untouched test partition. Because the classes are imbalanced, F1 and ROC-AUC are the primary metrics. Their mean is used only to select which complete pipeline is saved.

In [6]:
comparison, prediction_details = evaluate_trained_models(
    trained_models, data['X_test'], data['y_test']
)
save_comparison(comparison, results_dir / 'tables')
print('Final model comparison:')
print('F1 and ROC-AUC are primary; accuracy is secondary because the target is imbalanced.')
print(comparison.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

Final model comparison:
F1 and ROC-AUC are primary; accuracy is secondary because the target is imbalanced.
              model  accuracy  precision  recall     f1  roc_auc  primary_score
Logistic Regression    0.5744     0.3452  0.9982 0.5130   0.7615         0.6373
      Random Forest    0.7498     0.3735  0.1683 0.2320   0.7525         0.4923


## Confusion matrices

Each matrix shows real held-out predictions, making the false-positive and false-negative trade-off visible alongside the aggregate metrics.

In [7]:
confusion_paths = save_confusion_matrices(
    trained_models, prediction_details, results_dir / 'figures'
)
for path in confusion_paths:
    image = plt.imread(path)
    plt.figure(figsize=(7, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

C:\Users\Parth\AppData\Local\Temp\ipykernel_13088\3496938675.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save the best complete pipeline

The artifact includes both fitted preprocessing and the classifier, so later inference applies exactly the same transformations.

In [8]:
best_name, artifact_path = save_best_model(
    comparison, trained_models, PROJECT_ROOT / 'models'
)
summary = render_modeling_summary(
    audit, feature_counts, data, comparison, best_name, artifact_path, confusion_paths
)
summary_path = results_dir / 'modeling_evaluation_summary.txt'
summary_path.write_text(summary, encoding='utf-8')
print('Best model:', best_name)
print('Saved artifact:', artifact_path.resolve())
print('Saved summary:', summary_path.resolve())

Best model: Logistic Regression
Saved artifact: P:\College\Sem VII Acad\IPRM\Project\models\best_model.joblib
Saved summary: P:\College\Sem VII Acad\IPRM\Project\results\modeling_evaluation_summary.txt


## Phase 3 boundary

Model training and held-out evaluation are complete. SHAP, explainability analysis, and later business/adaptive experiments are intentionally deferred.